In [2]:
from mpetools import IslandTime, Segmentation
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from coastsatmaster.coastsat import SDS_preprocess, SDS_tools
from celluloid import Camera
import geopandas as gpd
import shapely
import pyproj

%load_ext autoreload
%autoreload 2

In [3]:
island = 'Vodamulaa'
country = 'Maldives'
island_info = IslandTime.retrieve_island_info(island, country, verbose=False)

lat = island_info['spatial_reference']['latitude']
lon = island_info['spatial_reference']['longitude']

files_to_ignore = ['2019-08-20-05-38-56_S2_Vodamulaa_Maldives_ms.tif',
                   ]
files_to_ignore = []


metadata = pickle.load(open(os.path.join(os.getcwd(), 'data', 'coastsat_data', '{}_{}'.format(island, country), '{}_{}_metadata.pkl'.format(island, country)), 'rb'))

# Retrieve settings
settings = island_info['timeseries_coastsat']['settings']

idx_sat = 0

# Reference shoreline
reference_shoreline = island_info['spatial_reference']['reference_shoreline']

# Loop in every satellite
for sat in ['S2']:
    print(sat)
    buff = island_info['spatial_reference']['reference_shoreline_buffer_{}'.format(sat)]
    buff_nan = np.where(buff == 0., np.nan, buff)

    # Empty metadata for this satellite
    if metadata[sat]['filenames'] == []:
        continue

    # File path and names from CoastSat folder
    filepath = SDS_tools.get_filepath(settings['inputs'], sat)
    filenames = metadata[sat]['filenames']

    fig, (ax1, ax2) = plt.subplots(1, 2)
    camera = Camera(fig)

    # Loop through images within the folder (taken from SDS_classify.py)
    for idx_img in range(len(filenames)):
        print('file # {}/{}'.format(idx_img+1, len(filenames)))

        if filenames[idx_img] in files_to_ignore:
            continue

        str_filename = filenames[idx_img].split('-')
        year = str_filename[0]
        month = str_filename[1]
        day = str_filename[2]
        
        # Get file name
        fn = SDS_tools.get_filenames(filenames[idx_img], filepath, sat)

        # Retrieve information about image
        try:
            im_ms, georef, cloud_mask, _, _, im_nodata = SDS_preprocess.preprocess_single(fn, sat, settings['cloud_mask_issue'], settings['pan_off'], 'C02')
        except:
            continue
        # Compute cloud_cover percentage (with no data pixels)
        cloud_cover_combined = np.divide(sum(sum(cloud_mask.astype(int))), (cloud_mask.shape[0] * cloud_mask.shape[1]))
        
        # If 99% of cloudy pixels in image -> skip
        if cloud_cover_combined > 0.99: 
            continue
            
        image_epsg = metadata[sat]['epsg'][idx_img]
    
        # transform lat/lon
        src_crs = pyproj.CRS('EPSG:4326')
        tgt_crs = pyproj.CRS('EPSG:{}'.format(image_epsg))
        transformer = pyproj.Transformer.from_crs(src_crs, tgt_crs, always_xy=True)

        xx, yy = transformer.transform(lon, lat)

        idx_x = int((xx - georef[0]) / georef[1])
        idx_y = int((yy - georef[3]) / georef[5])

        center_of_image = shapely.geometry.Point(idx_x, idx_y)

        # Remove no data pixels from the cloud mask (for example L7 bands of no data should not be accounted for)
        cloud_mask_adv = np.logical_xor(cloud_mask, im_nodata)

        # Compute updated cloud cover percentage (without no data pixels)
        cloud_cover = np.divide(sum(sum(cloud_mask_adv.astype(int))), (sum(sum((~im_nodata).astype(int)))))

        # Skip image if cloud cover is above threshold
        if cloud_cover > 0.15 or cloud_cover == 1: #
            continue

        # Get images
        img_red = im_ms[:, :, 2] * ~cloud_mask
        img_blue = im_ms[:, :, 0] * ~cloud_mask
        img_green = im_ms[:, :, 1] * ~cloud_mask 
        img_nir = im_ms[:, :, 3] * ~cloud_mask
        img_swir = im_ms[:, :, 4] * ~cloud_mask

        # NDVI = (NIR - RED) / (NIR + RED)
        img_NDVI = SDS_tools.nd_index(im_ms[:, :, 3], im_ms[:, :, 2], cloud_mask)

        # MNWDI = (GREEN - SWIR) / (GREEN + SWIR)
        img_MNDWI = SDS_tools.nd_index(im_ms[:, :, 4], im_ms[:, :, 1], cloud_mask)

        # Stacked array
        img_stacked = np.dstack((img_red, 
                                 img_green,
                                 img_blue, 
                                 img_nir,
                                 img_swir, 
                                 img_NDVI,
                                 img_MNDWI))

        rgb = np.dstack((img_red, img_green, img_blue))
        #wimg_masked = buff_nan * img_MNDWI

        try:
            segmented_image, cmap = Segmentation.unsupervised_classification(img_stacked, 7, plot_classification=False)
        
        except:
            continue
        
        labels = ['red', 'green', 'blue', 'NIR', 'SWIR', 'NDVI', 'MNDWI']
        labels = ['NIR', 'NDVI', 'MNDWI']

        for label, img_l in zip(labels, [img_nir, img_NDVI, img_MNDWI]):
            
            polygons_dict, X_peaks, X, histogram_distribution = Segmentation.segmentation(img_l, rgb, label, segmented_image, cmap, classes_to_consider=None, plot_results=False, animation=False)
            #Segmentation.associate_peaks_to_classification(img_l, segmented_image, label, X, histogram_distribution, X_peaks, cmap)

            # polygon
            try:
                polygons = polygons_dict['otsu']['polygons']
            except:
                continue

            # sort by area
            areas = [polygon.area for polygon in polygons]

            # largest polygon
            polygon = polygons[np.argmax(areas)]

            # test to see if polygon is the island
            # if not, take the second largest polygon
            
            if not polygon.contains(center_of_image):
                try:
                    polygon = polygons[np.argsort(areas)[-2]]
                except:
                    continue

            # plot polygon
            if label == 'NIR':
                gpd.GeoSeries(polygon).plot(ax=ax1, color='r')

        ax1.imshow(rgb, alpha=0.)
        ax1.text(img_l.shape[0]/3, img_l.shape[1]/5, '{}-{}-{}'.format(day, month, year), fontsize=15)  
        ax1.axis('off')
        ax2.imshow(rgb)
        plt.tight_layout()
        #ax.legend(fontsize=15)
        camera.snap()
        #plt.show()
    #plt.tight_layout()
    animation = camera.animate()
    animation.save('polygon_evolution_sentinel_Keredhdhoo.gif', writer='Pillow', fps=10)

S2
file # 1/408
file # 2/408
file # 3/408
file # 4/408
file # 5/408
file # 6/408
file # 7/408
file # 8/408
file # 9/408
file # 10/408
file # 11/408
file # 12/408
file # 13/408
file # 14/408
file # 15/408
file # 16/408
file # 17/408
file # 18/408
file # 19/408
file # 20/408
file # 21/408
file # 22/408
file # 23/408
file # 24/408
file # 25/408
file # 26/408
file # 27/408
file # 28/408
file # 29/408
file # 30/408
file # 31/408
file # 32/408
file # 33/408
file # 34/408
file # 35/408
file # 36/408
file # 37/408
file # 38/408
file # 39/408
file # 40/408
file # 41/408
file # 42/408
file # 43/408
file # 44/408
file # 45/408
file # 46/408
file # 47/408
file # 48/408
file # 49/408
file # 50/408
file # 51/408
file # 52/408
file # 53/408
file # 54/408
file # 55/408
file # 56/408
file # 57/408
file # 58/408
file # 59/408
file # 60/408
file # 61/408
file # 62/408
file # 63/408
file # 64/408
file # 65/408
file # 66/408
file # 67/408
file # 68/408
file # 69/408
file # 70/408
file # 71/408
file # 72/40

MovieWriter Pillow unavailable; using Pillow instead.


In [46]:
plt.imshow(segmented_image)

In [13]:
filenames[idx_img]

'2019-08-20-05-38-56_S2_Vodamulaa_Maldives_ms.tif'

In [98]:
# Create a meshgrid using numpy
x_values = np.linspace(extent[0], extent[1], num=img_red.shape[1])
y_values = np.linspace(extent[2], extent[3], num=img_red.shape[0])
x_mesh, y_mesh = np.meshgrid(x_values, y_values)

# Calculate Euclidean distances
distances = np.sqrt((x_mesh - xx)**2 + (y_mesh - yy)**2)

# Find the indices of the minimum distance
min_distance_index = np.unravel_index(np.argmin(distances), distances.shape)

# Get the closest point
closest_point = (x_mesh[min_distance_index], y_mesh[min_distance_index])

In [100]:
plt.imshow(img_red)
plt.scatter(min_distance_index[1], min_distance_index[0])
#plt.scatter(x_values, np.ones_like(x_values) * 50, c='r', s=0.1)

In [104]:
geotransform = georef

#extent = [transform[0], transform[0] + transform[1] * im_ms.shape[1], transform[3] + transform[5] * im_ms.shape[0], transform[3]]

In [105]:
x = int((xx - geotransform[0]) / geotransform[1])
y = int((yy - geotransform[3]) / geotransform[5])